# MLflow — Логирование экспериментов

Логируем оба эксперимента (Popularity Baseline и ALS) в MLflow для:
- Сравнения моделей
- Воспроизводимости экспериментов
- Версионирования артефактов

Запустить MLflow UI: `bash scripts/setup_mlflow.sh` → http://localhost:5000

In [1]:
import os
import sys
from pathlib import Path

import mlflow

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
MODELS_DIR   = Path(os.getenv("MODELS_DIR", PROJECT_ROOT / "models"))
MLFLOW_DIR   = PROJECT_ROOT / "mlruns"

# На Windows str(Path) даёт C:\..., что MLflow читает как схему "c:" — ошибка.
# as_uri() возвращает file:///C:/... — правильный формат.
mlflow.set_tracking_uri(MLFLOW_DIR.as_uri())
mlflow.set_experiment("ecommerce-recsys")

print(f"MLflow tracking URI: {mlflow.get_tracking_uri()}")
print(f"Директория MLflow:   {MLFLOW_DIR}")
print(f"Эксперимент:         ecommerce-recsys")

MLflow tracking URI: file:///C:/Users/denis/Projects/Study/ml-ecommerce-recsys/mlruns
Директория MLflow:   C:\Users\denis\Projects\Study\ml-ecommerce-recsys\mlruns
Эксперимент:         ecommerce-recsys


## 1. Popularity Baseline

In [2]:
with mlflow.start_run(run_name="popularity_baseline") as run:
    mlflow.log_params({
        "model_type": "popularity",
        "top_k": 10,
        "target": "addtocart",
        "split": "time_based_14d_holdout",
        "n_popular_fallback": 200,
    })

    mlflow.log_metrics({
        "recall_at_10": 0.0150,
        "ndcg_at_10": 0.0098,
        "precision_at_10": 0.0024,
    })

    artifact = MODELS_DIR / "artifact_popularity.pkl"
    if artifact.exists():
        mlflow.log_artifact(str(artifact), artifact_path="model")

    run_id_pop = run.info.run_id

print(f"Run ID (popularity): {run_id_pop}")
print("recall_at_10=0.0150 | ndcg_at_10=0.0098")

Run ID (popularity): 3df832cb486d474092d33b02c0cecda0
recall_at_10=0.0150 | ndcg_at_10=0.0098


## 2. ALS v1

In [3]:
with mlflow.start_run(run_name="als_v1") as run:
    mlflow.log_params({
        "model_type": "als",
        "factors": 64,
        "iterations": 20,
        "regularization": 0.05,
        "random_state": 42,
        "top_k": 10,
        "target": "addtocart",
        "weights": "view=1, addtocart=5, transaction=10",
        "split": "time_based_14d_holdout",
        "n_popular_fallback": 200,
        "library": "implicit==0.7.3",
    })

    mlflow.log_metrics({
        "recall_at_10": 0.0167,
        "ndcg_at_10": 0.0104,
        "precision_at_10": 0.0028,
    })

    for fname in ("als_model.pkl", "artifact.pkl"):
        path = MODELS_DIR / fname
        if path.exists():
            mlflow.log_artifact(str(path), artifact_path="model")

    run_id_als = run.info.run_id

print(f"Run ID (ALS v1): {run_id_als}")
print("recall_at_10=0.0167 | ndcg_at_10=0.0104")

Run ID (ALS v1): 118b97a573da421f81a7ec562600b5af
recall_at_10=0.0167 | ndcg_at_10=0.0104


## 3. Сравнение экспериментов

In [4]:
import mlflow
import pandas as pd

client = mlflow.tracking.MlflowClient()
exp = mlflow.get_experiment_by_name("ecommerce-recsys")
runs = mlflow.search_runs(experiment_ids=[exp.experiment_id])

cols = ["tags.mlflow.runName", "metrics.recall@10", "metrics.ndcg@10", "metrics.precision@10"]
print("Эксперименты:")
print(runs[[c for c in cols if c in runs.columns]].to_string(index=False))

Эксперименты:
tags.mlflow.runName
             als_v1
popularity_baseline
popularity_baseline


In [5]:
import mlflow
import pandas as pd

client = mlflow.tracking.MlflowClient()
exp = mlflow.get_experiment_by_name("ecommerce-recsys")
runs = mlflow.search_runs(experiment_ids=[exp.experiment_id])

cols = [
    "tags.mlflow.runName",
    "metrics.recall_at_10",
    "metrics.ndcg_at_10",
    "metrics.precision_at_10",
]
print("Эксперименты:")
print(runs[[c for c in cols if c in runs.columns]].to_string(index=False))

Эксперименты:
tags.mlflow.runName  metrics.recall_at_10  metrics.ndcg_at_10  metrics.precision_at_10
             als_v1                0.0167              0.0104                   0.0028
popularity_baseline                0.0150              0.0098                   0.0024
popularity_baseline                   NaN                 NaN                      NaN


## 4. Инструкции по воспроизведению

```bash
# Запустить MLflow UI
bash scripts/setup_mlflow.sh
# или
python -m mlflow ui --backend-store-uri ./mlruns

# Повторно залогировать эксперименты (после переобучения)
python scripts/log_mlflow_runs.py
```

**Лучшая модель: ALS v1**

| Метрика | Popularity | ALS v1 | Δ |
|---------|-----------|--------|---|
| recall_at_10 | 0.0150 | **0.0167** | +11.3% |
| ndcg_at_10 | 0.0098 | **0.0104** | +6.5% |
| precision_at_10 | 0.0024 | **0.0028** | +16.7% |

Низкие абсолютные значения ожидаемы: матрица имеет density 0.00068%,
97.4% пользователей в holdout имеют единственный addtocart-событие.
ALS даёт стабильное улучшение и высокое качество для пользователей с историей.